# Comment Classification: LLM Selection & Evaluation

Before running this notebook:
1. Install Ollama: https://ollama.com
2. To pull the models, run in terminal: 
    - ollama pull qwen3:8b
    - ollama pull llama3.1:8b

In [1]:
# importing libraries

import pandas as pd
import numpy as np
import ollama
import json
import re
from tqdm import tqdm
import krippendorff
from sklearn.metrics import classification_report, cohen_kappa_score

In [2]:
# loading the complete comments dataset
data_cpath = "data/comments/comments.csv"
data_comments = pd.read_csv(data_cpath, encoding="latin1")

# loading manually coded dataset for evaluation
data_epath = "data/comments/comments_eval.xlsx"
data_eval = pd.read_excel(data_epath, engine="openpyxl")


Here, we collapse the `breadth` variable.  

In [3]:
data_eval["breadth"] = data_eval["breadth"].clip(upper=3)

In [4]:
# defining the system prompt inline with the codebook

with open("data/comments/system_prompt.txt", "r", encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

In [5]:
# define classifier function with qwen3:8b set as default

def classify_comment(text, parent_text=None, model="qwen3:8b"):
    """Classify a single comment on all 6 codebook variables in one call.
    Returns a dict of 6 ints, or a dict of -1s if parsing fails."""
    fallback = {"pers_exp": -1, "emot_exp": -1, "pol_opin": -1,
                "breadth": -1, "valence": -1, "contr": -1}

    if not isinstance(text, str) or text.strip() == "":
        return {"pers_exp": 0, "emot_exp": 0, "pol_opin": 0,
                "breadth": 0, "valence": 4, "contr": 0}

    user_msg = f"Comment: {text}"
    if parent_text:
        user_msg += f"\n\n(This is a reply to the following parent comment: {parent_text})"

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_msg}
        ],
        options={"temperature": 0}
    )

    out = response["message"]["content"].strip()

    match = re.search(r"\{.*\}", out, re.DOTALL)
    if not match:
        return fallback

    try:
        parsed = json.loads(match.group(0))
    except json.JSONDecodeError:
        return fallback

    result = {}
    bounds = {"pers_exp": (0,1), "emot_exp": (0,1), "pol_opin": (0,1),
              "breadth": (0,3), "valence": (1,4), "contr": (0,1)}
    for key, (lo, hi) in bounds.items():
        val = parsed.get(key, -1)
        try:
            val = int(val)
        except (TypeError, ValueError):
            val = -1
        result[key] = val if lo <= val <= hi else -1

    return result

In [6]:
# post_number resets per topic, so it's not unique on its own - build a composite key instead
def make_id(topic, post_number):
    return f"{topic}_{int(post_number)}"

# build ids on the FULL data_comments first, so parent lookups still work even for
# replies whose parent falls outside whatever subset we classify below
data_comments["row_id"] = data_comments.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
data_eval["row_id"] = data_eval.apply(lambda r: make_id(r["topic"], r["post_number"]), axis=1)
id_col = "row_id"

reply_col = "reply_to_post_number" if "reply_to_post_number" in data_comments.columns else None
text_by_id = dict(zip(data_comments["row_id"], data_comments["raw"].fillna("")))

def get_parent_text(row):
    if reply_col and pd.notna(row.get(reply_col)):
        parent_id = make_id(row["topic"], row[reply_col])
        return text_by_id.get(parent_id)
    return None

data_comments_full = data_comments.copy()  # keep the unfiltered version for the full run later

# for now, only classify comments that already have a manual code (validation pass) -
# comment this line out later to run the full dataset instead
data_comments = data_comments[data_comments[id_col].isin(data_eval[id_col])].copy()
print(f"Classifying {len(data_comments)} comments")

Classifying 150 comments


In [7]:
MODELS = ["qwen3:8b", "llama3.1:8b"]

In [8]:
VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
texts = data_comments["raw"].fillna("")


# delete:
#data_comments = data_comments.head().copy()


for model_name in MODELS:
    results = []
    for i, row in tqdm(data_comments.iterrows(), total=len(data_comments), desc=f"Classifying ({model_name})"):
        parent_text = get_parent_text(row)
        codes = classify_comment(texts.loc[i], parent_text=parent_text, model=model_name)
        results.append(codes)

    results_df = pd.DataFrame(results)
    model_tag = model_name.replace(":", "_").replace(".", "_")
    pred_cols = [f"{v}_{model_tag}" for v in VARS]
    data_comments[pred_cols] = results_df.values

Classifying (llama3.1:8b): 100%|██████████| 150/150 [28:19<00:00, 11.33s/it]


In [9]:
data_comments.head()

,coder,post_number,user,topic,raw,pers_exp,emot_exp,pol_opin,breadth,valence,...,pol_opin_qwen3_8b,breadth_qwen3_8b,valence_qwen3_8b,contr_qwen3_8b,pers_exp_llama3_1_8b,emot_exp_llama3_1_8b,pol_opin_llama3_1_8b,breadth_llama3_1_8b,valence_llama3_1_8b,contr_llama3_1_8b
12,NaN,19,Maksimo,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Ich stimme dem 9 Punkteplan zu in allen Bereic...,NaN,NaN,NaN,NaN,NaN,...,1,2,1,1,0,1,1,2,2,1
27,NaN,34,brunolina,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Spontan - ohne jetzt auf die einzelnen Inhalte...,NaN,NaN,NaN,NaN,NaN,...,1,1,2,1,-1,-1,-1,-1,-1,-1
29,NaN,36,holzwurmpaul,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Überfällig ja aber wer hat soviel Eier in der ...,NaN,NaN,NaN,NaN,NaN,...,1,1,2,1,1,1,1,3,2,1
49,NaN,56,Sudiko,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Also ich finde auch das die Bundeswehr im Inne...,NaN,NaN,NaN,NaN,NaN,...,1,3,3,1,0,1,1,3,2,1
57,NaN,64,Ewald,Welche Meinung haben Sie zum 9-Punkte-Plan? (190),Vom Ansatz her richtig. Auch eine Einbindung d...,NaN,NaN,NaN,NaN,NaN,...,1,3,3,1,0,0,1,3,2,1


In [10]:
# create report

VARS = ["pers_exp", "emot_exp", "pol_opin", "breadth", "valence", "contr"]
var_weights = {"pers_exp": None, "emot_exp": None, "pol_opin": None, "contr": None,
               "breadth": "linear", "valence": None}
var_levels = {"pers_exp": "nominal", "emot_exp": "nominal", "pol_opin": "nominal",
              "contr": "nominal", "breadth": "interval", "valence": "nominal"}

def krippendorff_alpha(y_true, y_pred, level):
    try:
        data = np.array([y_true, y_pred], dtype=float)
        return krippendorff.alpha(reliability_data=data, level_of_measurement=level)
    except (ZeroDivisionError, ValueError):
        return float("nan")

data_eval_renamed = data_eval.rename(columns={v: f"{v}_true" for v in VARS})
merged = data_eval_renamed.merge(data_comments, on=id_col)
print(f"Matched {len(merged)} of {len(data_eval)} rows\n")

all_rows = []

for model_name in MODELS:
    model_tag = model_name.replace(":", "_").replace(".", "_")
    print(f"\n########## {model_name} ##########")
    for var in VARS:
        y_true = merged[f"{var}_true"]
        y_pred = merged[f"{var}_{model_tag}"]

        mask = y_true.notna() & y_pred.notna() & (y_pred != -1)
        y_true_m, y_pred_m = y_true[mask], y_pred[mask]

        print(f"=== {var} (n={mask.sum()}) ===")
        report_dict = classification_report(y_true_m, y_pred_m, zero_division=0, output_dict=True)
        print(classification_report(y_true_m, y_pred_m, zero_division=0))

        kappa = cohen_kappa_score(y_true_m, y_pred_m, weights=var_weights[var])
        alpha = krippendorff_alpha(y_true_m.values, y_pred_m.values, var_levels[var])
        print(f"Cohen's Kappa: {kappa:.3f}   Krippendorff's alpha ({var_levels[var]}): {alpha:.3f}\n")

        for class_label, metrics in report_dict.items():
            if isinstance(metrics, dict):  # skips "accuracy", which is a bare float
                all_rows.append({
                    "model": model_name, "variable": var, "n": mask.sum(),
                    "class": class_label,
                    "precision": metrics["precision"], "recall": metrics["recall"],
                    "f1_score": metrics["f1-score"], "support": metrics["support"],
                    "accuracy": round(report_dict["accuracy"], 3),
                    "cohens_kappa": round(kappa, 3),
                    "krippendorff_alpha": round(alpha, 3) if not np.isnan(alpha) else "n/a",
                })

report_df = pd.DataFrame(all_rows)
report_df.to_excel("data/comments/eval_report.xlsx", index=False)
print("Saved eval_report.xlsx")

Matched 150 of 150 rows


########## qwen3:8b ##########
=== pers_exp (n=150) ===
              precision    recall  f1-score   support

           0       0.96      0.97      0.96       132
           1       0.75      0.67      0.71        18

    accuracy                           0.93       150
   macro avg       0.85      0.82      0.83       150
weighted avg       0.93      0.93      0.93       150

Cohen's Kappa: 0.668   Krippendorff's alpha (nominal): 0.669

=== emot_exp (n=150) ===
              precision    recall  f1-score   support

           0       0.95      0.83      0.88       111
           1       0.64      0.87      0.74        39

    accuracy                           0.84       150
   macro avg       0.79      0.85      0.81       150
weighted avg       0.87      0.84      0.85       150

Cohen's Kappa: 0.628   Krippendorff's alpha (nominal): 0.625

=== pol_opin (n=150) ===
              precision    recall  f1-score   support

           0       1.00      0.28  